# Train Qwen3 with GRPO on a Google Colab GPU

This self-contained notebook trains LoRA adapters for **Qwen/Qwen3-0.6B**
using GSM8K and a verified final-integer reward. No repository clone is needed.

1. Select **Runtime → Change runtime type → GPU** (for example, a T4 if offered).
2. Run cells in order. The default is a small smoke run; switch `SMOKE_TEST`
   to `False` for the longer learning experiment.
3. Inspect results, reload the adapter, and download its ZIP. Hugging Face
   publication and Google Drive backup are optional and disabled initially.

Colab GPU availability and session duration vary. `/content` storage is
temporary: download or back up results before the runtime is deleted.
A few training steps demonstrate the machinery, not improved reasoning.
This notebook has local structural checks and shares the CPU-tested training
implementation; end-to-end execution on a Colab GPU has not been verified.


## 1. Install the training stack

Keep Colab's existing CUDA-enabled PyTorch. The pinned TRL, Transformers and
PEFT versions match this implementation. Training and model loading run in
fresh Python subprocesses so an older library imported in the notebook
kernel does not need to be reused after installation.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

WORK_DIR = Path("/content/grpo_qwen")
WORK_DIR.mkdir(parents=True, exist_ok=True)

REQUIREMENTS = "# A separate training environment avoids changing the repository's uv.lock.\n# Install CPU or CUDA PyTorch first for your device (see GRPO_CPU.md / practice guide).\ntrl==0.22.2\ntransformers==4.56.2\npeft==0.17.1\ndatasets>=4.0,<5\naccelerate>=1.4,<2\n"
requirements_path = WORK_DIR / "requirements-grpo.txt"
requirements_path.write_text(REQUIREMENTS, encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)], check=True)
print("Training dependencies installed. Continue to the GPU check.")


## 2. Verify that the runtime has a CUDA GPU

This intentionally stops if Colab is still using a CPU runtime. A T4 uses
fp16 here; GPUs with bf16 support use bf16. Both modes train LoRA weights.


In [ ]:
gpu_check = """
import torch
import transformers, trl, peft
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then rerun."
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Precision:", "bf16" if torch.cuda.is_bf16_supported() else "fp16")
print("Versions:", torch.__version__, transformers.__version__, trl.__version__, peft.__version__)
"""
import textwrap
subprocess.run([sys.executable, "-c", textwrap.dedent(gpu_check)], check=True)


## 3. Write the training implementation

The next cell embeds the complete project script, including answer parsing,
model loading, GRPO optimization, before/after evaluation and adapter saving.
Expand it to study the implementation. The demo checks reward arithmetic
without downloading the model or dataset.


In [ ]:
GRPO_SOURCE = r'''"""Train Qwen on GSM8K with GRPO and verifiable integer-answer rewards.

See GRPO_practice.md for setup, the objective, and worked exercises.
Imports are lazy: --help, --demo, and reward tests need only Python.
"""
from __future__ import annotations

import argparse
import csv
import json
import math
import os
from pathlib import Path
import re
from statistics import mean, stdev

INSTRUCTION = r"Solve the problem step by step. End with one final integer answer in \boxed{...}."
INTEGER = r"[+-]?(?:\d{1,3}(?:,\d{3})+|\d+)"


def extract_hash_answer(text):
    """Parse GSM8K's final #### answer; reject malformed labels early."""
    match = re.search(r"####\s*(" + INTEGER + r")\s*$", text)
    if match is None:
        raise ValueError(f"Invalid GSM8K answer: {text!r}")
    return int(match.group(1).replace(",", ""))


def extract_boxed_int(text):
    """Use the last box; decimals, expressions and malformed commas earn zero."""
    # Include nested/invalid box starts so an earlier valid box cannot hide one.
    starts = list(re.finditer(r"\\boxed\s*\{", text))
    if not starts:
        return None
    tail = text[starts[-1].end():]
    raw, separator, _ = tail.partition("}")
    if not separator or re.fullmatch(INTEGER, raw.strip()) is None:
        return None
    return int(raw.strip().replace(",", ""))


def completion_text(completion):
    """TRL supports plain text and conversational assistant messages."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        return "\n".join(message["content"] for message in completion
                         if message.get("role") == "assistant")
    raise TypeError("Expected text or a list of chat messages")


def reward_func(completions, answer, **kwargs):
    """One outcome reward per sampled completion; labels never enter prompts."""
    if len(completions) != len(answer):
        raise ValueError("Each completion must have one ground-truth answer")
    rewards = []
    for completion, target in zip(completions, answer):
        predicted = extract_boxed_int(completion_text(completion))
        rewards.append(float(predicted is not None and predicted == int(target)))
    return rewards


def group_advantages(rewards, epsilon=1e-4):
    """Teaching helper matching TRL's sample-std group normalization.

    TRL performs the actual tensor calculation and gradient updates.
    Equal-reward groups have zero task-reward learning signal.
    """
    if len(rewards) < 2:
        raise ValueError("GRPO needs at least two completions per prompt")
    baseline, spread = mean(rewards), stdev(rewards)
    return [(reward - baseline) / (spread + epsilon) for reward in rewards]


def prepare_example(example, tokenizer):
    messages = [{"role": "user", "content": example["question"] + "\n" + INSTRUCTION}]
    # Render once so training and evaluation use the same non-thinking template.
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    return {"prompt": prompt, "answer": extract_hash_answer(example["answer"])}


def load_dataset(tokenizer, train_size=256, eval_size=32, seed=42):
    from datasets import load_dataset as hf_load_dataset

    splits = []
    for split, limit in (("train", train_size), ("test", eval_size)):
        dataset = hf_load_dataset("openai/gsm8k", "main", split=split).shuffle(seed=seed)
        dataset = dataset.select(range(min(limit, len(dataset))))
        dataset = dataset.map(lambda row: prepare_example(row, tokenizer),
                              remove_columns=dataset.column_names)
        splits.append(dataset)
    return tuple(splits)


def load_library(backend="transformers", device="cuda"):
    """Load Unsloth first when requested so its patches precede TRL imports."""
    if device == "cpu" and backend == "unsloth":
        raise ValueError("CPU training uses --backend transformers; Unsloth requires a GPU.")
    try:
        if backend == "unsloth":
            import unsloth  # noqa: F401
        import torch
        from trl import GRPOConfig, GRPOTrainer
        import peft  # noqa: F401
    except ImportError as error:
        raise RuntimeError("Missing training dependencies; see GRPO_practice.md setup.") from error
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. Use --device cpu for CPU training.")
    return torch, GRPOConfig, GRPOTrainer


def load_model(model_name, backend="transformers", max_seq_length=1536, seed=42,
               device="cuda", lora_rank=32):
    import torch

    if backend == "unsloth":
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name, max_seq_length=max_seq_length,
            load_in_4bit=False, full_finetuning=False,
        )
        model = FastLanguageModel.get_peft_model(
            model, r=lora_rank, lora_alpha=2 * lora_rank,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
            use_gradient_checkpointing="unsloth", random_state=seed,
        )
    else:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from peft import LoraConfig, get_peft_model
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name, dtype=(torch.float32 if device == "cpu" else
                                     torch.bfloat16 if torch.cuda.is_bf16_supported()
                                     else torch.float16),
        )
        model = get_peft_model(model, LoraConfig(
            r=lora_rank, lora_alpha=2 * lora_rank, lora_dropout=0.0, bias="none",
            task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
        ))
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    model.config.pad_token_id = tokenizer.pad_token_id
    return model, tokenizer


class GRPO:
    """Coordinate rollouts, rewards, optimization and held-out evaluation."""

    reward_func = staticmethod(reward_func)

    def __init__(self, options):
        self.options = options
        torch, config_class, trainer_class = load_library(options.backend, options.device)
        from transformers import set_seed
        set_seed(options.seed)
        if options.device == "cpu":
            torch.set_num_threads(options.cpu_threads)
        self.model, self.tokenizer = load_model(
            options.model, options.backend,
            options.max_prompt_length + options.max_completion_length, options.seed,
            device=options.device, lora_rank=options.lora_rank,
        )
        self.ds_train, self.ds_test = load_dataset(
            self.tokenizer, options.train_size, options.eval_size, options.seed,
        )
        self.output_dir = Path(options.output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        cpu = options.device == "cpu"
        bf16 = not cpu and torch.cuda.is_bf16_supported()
        training_args = config_class(
            output_dir=str(self.output_dir), learning_rate=5e-6,
            adam_beta1=0.9, adam_beta2=0.99, weight_decay=0.1,
            # A one-step smoke run must not spend its only update at zero LR.
            warmup_ratio=0.0 if options.max_steps == 1 else 0.1,
            lr_scheduler_type="cosine", logging_steps=1,
            use_cpu=cpu, bf16=bf16, fp16=not cpu and not bf16,
            optim="adamw_torch", dataloader_pin_memory=not cpu,
            per_device_train_batch_size=options.per_device_train_batch_size,
            gradient_accumulation_steps=options.gradient_accumulation_steps,
            num_generations=options.num_generations,
            max_prompt_length=options.max_prompt_length,
            max_completion_length=options.max_completion_length,
            max_steps=options.max_steps, save_steps=options.max_steps,
            report_to="none", use_vllm=False, remove_unused_columns=False,
            temperature=1.0, top_p=0.95, seed=options.seed,
            # Explicit choices teach original GRPO instead of changing TRL defaults.
            beta=options.beta, epsilon=0.2, loss_type="grpo", scale_rewards="group",
            num_iterations=options.num_iterations, gradient_checkpointing=True,
            # TRL 0.22 re-enables reentrant checkpointing after generation;
            # use that mode consistently so it also enables input gradients.
            gradient_checkpointing_kwargs={"use_reentrant": True},
        )
        self.trainer = trainer_class(
            model=self.model, processing_class=self.tokenizer,
            reward_funcs=[reward_func], args=training_args, train_dataset=self.ds_train,
        )

    def training(self):
        """TRL handles old-policy log-probs, clipping, reference KL and backprop."""
        result = self.trainer.train()
        self.trainer.save_model(str(self.output_dir / "adapter"))
        self.tokenizer.save_pretrained(str(self.output_dir / "adapter"))
        self.trainer.save_state()
        history = self.trainer.state.log_history
        (self.output_dir / "log_history.json").write_text(
            json.dumps(history, indent=2), encoding="utf-8",
        )
        rows = [entry for entry in history if "reward" in entry]
        with (self.output_dir / "rewards.csv").open("w", newline="", encoding="utf-8") as handle:
            writer = csv.writer(handle)
            writer.writerow(["step", "reward", "moving_average_5"])
            for index, row in enumerate(rows):
                window = rows[max(0, index - 4):index + 1]
                writer.writerow([row["step"], row["reward"], mean(x["reward"] for x in window)])
        return result

    def evaluation(self, name="after"):
        """Greedy exact-match accuracy on the same held-out subset before/after."""
        import torch
        if self.options.backend == "unsloth":
            from unsloth import FastLanguageModel
            FastLanguageModel.for_inference(self.model)
        was_training = self.model.training
        self.model.eval()
        rows = []
        try:
            with torch.inference_mode():
                for example in self.ds_test:
                    inputs = self.tokenizer(
                        example["prompt"], return_tensors="pt", truncation=True,
                        max_length=self.options.max_prompt_length, add_special_tokens=False,
                        return_token_type_ids=False,
                    ).to(next(self.model.parameters()).device)
                    output = self.model.generate(
                        **inputs, max_new_tokens=self.options.max_completion_length,
                        do_sample=False, pad_token_id=self.tokenizer.pad_token_id,
                        eos_token_id=self.tokenizer.eos_token_id, use_cache=True,
                    )
                    # Score only new tokens; prompt text must never supply the answer.
                    text = self.tokenizer.decode(output[0, inputs["input_ids"].shape[1]:],
                                                 skip_special_tokens=True)
                    predicted = extract_boxed_int(text)
                    rows.append({"answer": example["answer"], "prediction": predicted,
                                 "correct": predicted == example["answer"], "completion": text})
        finally:
            if self.options.backend == "unsloth":
                FastLanguageModel.for_training(self.model)
            self.model.train(was_training)
        metrics = {"accuracy": mean(row["correct"] for row in rows),
                   "format_rate": mean(row["prediction"] is not None for row in rows),
                   "examples": len(rows)}
        (self.output_dir / f"evaluation_{name}.json").write_text(
            json.dumps({"metrics": metrics, "predictions": rows}, indent=2), encoding="utf-8",
        )
        print(f"{name}: {metrics}")
        return metrics


def Huggingface_upload(experiment, repo_id):
    """Publish adapters only when --push-to-hub is explicitly supplied."""
    experiment.model.push_to_hub(repo_id)
    experiment.tokenizer.push_to_hub(repo_id)


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--demo", action="store_true", help="Show group rewards without downloads")
    parser.add_argument("--backend", choices=["transformers", "unsloth"], default="transformers")
    parser.add_argument("--device", choices=["cuda", "cpu"], default="cuda",
                        help="CPU selects smaller defaults and float32 training")
    parser.add_argument("--model", default="Qwen/Qwen3-0.6B")
    parser.add_argument("--output-dir", default=None)
    parser.add_argument("--train-size", type=int, default=None)
    parser.add_argument("--eval-size", type=int, default=None)
    parser.add_argument("--max-steps", type=int, default=None)
    parser.add_argument("--num-generations", type=int, default=None)
    parser.add_argument("--max-prompt-length", type=int, default=None)
    parser.add_argument("--max-completion-length", type=int, default=None)
    parser.add_argument("--per-device-train-batch-size", type=int)
    parser.add_argument("--gradient-accumulation-steps", type=int)
    parser.add_argument("--num-iterations", type=int)
    parser.add_argument("--lora-rank", type=int)
    parser.add_argument("--cpu-threads", type=int, default=min(4, os.cpu_count() or 1))
    parser.add_argument("--beta", type=float, default=0.04)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--push-to-hub", metavar="USER/REPO")
    args = parser.parse_args(argv)
    defaults = {
        "output_dir": ("GRPO_output", "GRPO_output_cpu"),
        "train_size": (256, 16), "eval_size": (32, 2), "max_steps": (50, 5),
        "num_generations": (8, 2), "max_prompt_length": (1024, 512),
        "max_completion_length": (512, 128), "per_device_train_batch_size": (2, 1),
        "gradient_accumulation_steps": (4, 2), "num_iterations": (2, 1),
        "lora_rank": (32, 8),
    }
    for field, values in defaults.items():
        if getattr(args, field) is None:
            setattr(args, field, values[1 if args.device == "cpu" else 0])
    if args.device == "cpu" and args.backend == "unsloth":
        parser.error("--device cpu requires --backend transformers")
    for field in ("train_size", "eval_size", "max_steps", "max_prompt_length",
                  "max_completion_length", "per_device_train_batch_size",
                  "gradient_accumulation_steps", "num_iterations", "lora_rank", "cpu_threads"):
        if getattr(args, field) <= 0:
            parser.error(f"--{field.replace('_', '-')} must be positive")
    effective_batch = (args.per_device_train_batch_size * args.gradient_accumulation_steps
                       * int(os.environ.get("WORLD_SIZE", "1")))
    if args.num_generations < 2 or effective_batch % args.num_generations:
        parser.error(f"--num-generations must be >=2 and divide effective batch {effective_batch}")
    if not math.isfinite(args.beta) or args.beta < 0:
        parser.error("--beta must be finite and nonnegative")
    return args


def main(argv=None):
    options = parse_args(argv)
    if options.demo:
        completions = [r"2 + 2 = 4. \boxed{4}", r"\boxed{5}", "4", r"\boxed{4}"]
        rewards = reward_func(completions, [4] * 4)
        print("Rewards:", rewards)
        print("Group-relative advantages:", [round(x, 4) for x in group_advantages(rewards)])
        print("Equal-reward group:", group_advantages([0.0] * 4))
        return
    print(f"Training on {options.device}: {options.max_steps} steps, "
          f"{options.num_generations} completions per question, "
          f"adapter output: {Path(options.output_dir) / 'adapter'}", flush=True)
    experiment = GRPO(options)
    experiment.evaluation("before")
    experiment.training()
    experiment.evaluation("after")
    if options.push_to_hub:
        Huggingface_upload(experiment, options.push_to_hub)


if __name__ == "__main__":
    main()
'''
SCRIPT = WORK_DIR / "GRPO.py"
SCRIPT.write_text(GRPO_SOURCE, encoding="utf-8")
subprocess.run([sys.executable, str(SCRIPT), "--demo"], check=True)


## 4. Choose training settings

Four completions are sampled for each question. Microbatch 1 × accumulation 4
gives a generation batch of four completions on this single-GPU runtime.
Original GRPO uses group-relative rewards and a KL penalty to the initial model.

Start with `SMOKE_TEST=True`. After it succeeds, set it to `False` and rerun
this cell and the cells below for a 25-step experiment in a different folder.
Short smoke completions often stop before the final answer. Increase the
completion limit for learning experiments if many responses are truncated.


In [ ]:
SMOKE_TEST = True
MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_STEPS = 2 if SMOKE_TEST else 25
TRAIN_SIZE = 8 if SMOKE_TEST else 256
EVAL_SIZE = 2 if SMOKE_TEST else 16
MAX_PROMPT_LENGTH = 512
MAX_COMPLETION_LENGTH = 64 if SMOKE_TEST else 256
NUM_GENERATIONS = 4
MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LORA_RANK = 16
BETA = 0.04
SEED = 42
OUTPUT_DIR = WORK_DIR / ("qwen3_grpo_smoke" if SMOKE_TEST else "qwen3_grpo")
ADAPTER_DIR = OUTPUT_DIR / "adapter"

assert NUM_GENERATIONS >= 2
assert (MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS) % NUM_GENERATIONS == 0
run_settings = {
    "model": MODEL_NAME, "device": "cuda", "backend": "transformers",
    "max_steps": MAX_STEPS, "train_size": TRAIN_SIZE, "eval_size": EVAL_SIZE,
    "max_prompt_length": MAX_PROMPT_LENGTH, "max_completion_length": MAX_COMPLETION_LENGTH,
    "num_generations": NUM_GENERATIONS, "per_device_train_batch_size": MICRO_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "lora_rank": LORA_RANK, "beta": BETA, "seed": SEED, "num_iterations": 1,
}
command = [sys.executable, "-u", str(SCRIPT)]
for key, value in run_settings.items():
    command.extend(["--" + key.replace("_", "-"), str(value)])
command.extend(["--output-dir", str(OUTPUT_DIR)])
print("Mode:", "smoke test" if SMOKE_TEST else "learning experiment")
print("Adapter will be saved to:", ADAPTER_DIR)


## 5. Train and evaluate

The first run downloads Qwen and GSM8K. Logs are streamed below. Evaluation
uses the same held-out subset before and after training and scores only new
response tokens. The adapter is saved before the final evaluation starts.

If you get CUDA out-of-memory, reduce the completion limit first, keep
microbatch 1, and try two generations with accumulation 2. If the process
was interrupted or lost GPU memory, restart the runtime before retrying.
This notebook starts a new training run; it does not automatically resume
an interrupted checkpoint. Reusing an output folder can overwrite results.


In [ ]:
import json
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "run_settings.json").write_text(json.dumps(run_settings, indent=2), encoding="utf-8")
child_environment = os.environ.copy()
child_environment["TOKENIZERS_PARALLELISM"] = "false"
subprocess.run(command, check=True, env=child_environment, cwd=str(WORK_DIR))
assert (ADAPTER_DIR / "adapter_config.json").is_file()
assert (ADAPTER_DIR / "adapter_model.safetensors").is_file()
print("Training completed. Adapter:", ADAPTER_DIR)


## 6. Inspect results and prepare a model card

Accuracy is exact final-integer correctness under the boxed-answer contract.
`format_rate` measures whether a valid boxed integer was generated.
A correct final number does not verify intermediate reasoning. Small test
subsets are noisy; avoid repeatedly tuning on the test split.


In [ ]:
import csv
summary = {"settings": run_settings}
for stage in ("before", "after"):
    record = json.loads((OUTPUT_DIR / f"evaluation_{stage}.json").read_text())
    summary[stage] = record["metrics"]
    print(stage, record["metrics"])
print("\nExample response after training:")
print(record["predictions"][0]["completion"])
with (OUTPUT_DIR / "rewards.csv").open() as handle:
    reward_rows = list(csv.DictReader(handle))
print("\nLast reward records:", reward_rows[-5:])
(ADAPTER_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

card = f"""---
base_model: {MODEL_NAME}
library_name: peft
license: apache-2.0
datasets:
- openai/gsm8k
tags:
- grpo
- lora
- reinforcement-learning
---
# Qwen3 GSM8K GRPO adapter

A LoRA adapter for `{MODEL_NAME}`, trained on a shuffled GSM8K training subset
with a verifiable boxed-integer outcome reward. The original base weights
are required to load this adapter. See `training_summary.json` for settings
and before/after results, including sample counts.

This is an educational {"smoke run" if SMOKE_TEST else "training experiment"}.
Results do not establish general reasoning improvements. Exact-answer scoring
does not verify intermediate reasoning, and the held-out subset is small.
The base model is provided by Qwen under Apache-2.0.
"""
(ADAPTER_DIR / "README.md").write_text(textwrap.dedent(card), encoding="utf-8")
print("\nFiles ready to share:", sorted(path.name for path in ADAPTER_DIR.iterdir()))


## 7. Reload the saved adapter and generate an answer

This fresh process checks that the adapter works with its original base
model. The training process has exited, so its GPU model memory is released.


In [ ]:
inference_code = r"""
import sys
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
adapter = sys.argv[1]
tokenizer = AutoTokenizer.from_pretrained(adapter)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoPeftModelForCausalLM.from_pretrained(adapter, dtype=dtype).to("cuda")
model.eval()
messages = [{"role": "user", "content":
    r"A box holds 6 apples. How many apples are in 3 boxes? Reason step by step and end in \boxed{...}."}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                       add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False,
                   return_token_type_ids=False).to("cuda")
with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                            pad_token_id=tokenizer.pad_token_id,
                            eos_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))
"""
subprocess.run([sys.executable, "-c", textwrap.dedent(inference_code), str(ADAPTER_DIR)], check=True)


## 8. Download the adapter ZIP

The ZIP contains adapter weights, configuration, tokenizer, model card and
training summary. It contains no full base-model weights. Download it before
ending the Colab session. Extract it before uploading the folder to the Hub.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(WORK_DIR / (OUTPUT_DIR.name + "_adapter")),
                              "zip", root_dir=str(ADAPTER_DIR))
print("Download:", archive)
files.download(archive)


## 9. Optional: upload the adapter to Hugging Face

Set `UPLOAD_TO_HUB=True` and replace the repository ID. This explicitly creates
a **private** repository if it does not exist and uploads only the adapter
folder. Existing repository visibility is not changed. Enter a write-capable
token in the hidden prompt; never paste it into a code cell. Review the card
and results before changing the repository to public.


In [ ]:
UPLOAD_TO_HUB = False
REPO_ID = "YOUR_USERNAME/qwen3-0.6b-grpo-gsm8k"
if UPLOAD_TO_HUB:
    assert "YOUR_USERNAME" not in REPO_ID and REPO_ID.count("/") == 1
    import getpass
    from huggingface_hub import HfApi
    token = getpass.getpass("Hugging Face write token: ")
    api = HfApi(token=token)
    try:
        api.create_repo(repo_id=REPO_ID, repo_type="model", private=True, exist_ok=True)
        api.upload_folder(repo_id=REPO_ID, repo_type="model", folder_path=str(ADAPTER_DIR),
                          commit_message="Upload Qwen3 GSM8K GRPO LoRA adapter")
        print("Uploaded:", "https://huggingface.co/" + REPO_ID)
    finally:
        del api, token
else:
    print("Upload disabled. The local adapter and ZIP are ready.")


## 10. Optional: back up the full run to Google Drive

Enable this to preserve checkpoints, evaluation predictions and logs as well
as the adapter. It mounts your Drive interactively and writes a ZIP to
`MyDrive/GRPO_backups/`. Downloads and Drive backup are independent of Hub upload.


In [ ]:
BACKUP_TO_DRIVE = False
if BACKUP_TO_DRIVE:
    from datetime import datetime, timezone
    from google.colab import drive
    drive.mount("/content/drive")
    backup_dir = Path("/content/drive/MyDrive/GRPO_backups")
    backup_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
    backup = shutil.make_archive(str(WORK_DIR / (OUTPUT_DIR.name + "_" + timestamp)),
                                 "zip", root_dir=str(OUTPUT_DIR))
    destination = shutil.copy2(backup, backup_dir)
    print("Full-run backup:", destination)
else:
    print("Drive backup disabled.")


## Learn from this run

- For rewards `[1, 0, 0, 1]`, the sample-standard-deviation normalized
  advantages are about `[+0.866, -0.866, -0.866, +0.866]`.
- If all completions for a question get the same reward, their task-reward
  advantages are zero. Check outputs and completion limits before adding steps.
- Compare final-answer accuracy and valid-answer rate separately. A format
  improvement is useful, but does not by itself demonstrate better arithmetic.

References: [Colab resource limits](https://research.google.com/colaboratory/faq.html),
[Qwen3 model card](https://huggingface.co/Qwen/Qwen3-0.6B),
[versioned TRL implementation](https://github.com/huggingface/trl/tree/v0.22.2),
[Hugging Face uploads](https://huggingface.co/docs/huggingface_hub/guides/upload).
